In [0]:

# %pip uninstall -y cudf-cu13 cuml-cu13 cugraph-cu13 cuvs-cu13 libcuml-cu13 libcuvs-cu13 libraft-cu13 pylibcudf-cu13 pylibraft-cu13 rmm-cu13

# Install PyTorch with CUDA 12.9
%pip install --upgrade --no-cache-dir \
  --index-url https://download.pytorch.org/whl/cu129 \
  "torch==2.9.1" "torchvision==0.24.1" "torchaudio==2.9.1"

# Install CUDA-12 RAPIDS packages, pinned to the same RAPIDS release
%pip install --upgrade --no-cache-dir \
  "cupy-cuda12x>=13.6.0" \
  "rmm-cu12==26.02.*" \
  "cudf-cu12==26.02.*" \
  "cuml-cu12==26.02.*"

dbutils.library.restartPython()

In [0]:
import cuml
%reload_ext cuml.accel

In [0]:

import time
import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
pio.renderers.default = "browser"

from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, adjusted_rand_score
from multiprocessing import Manager
from cuml.manifold import UMAP
from cuml.cluster import KMeans  # GPU-accelerated KMeans
from pathlib import Path
import numpy as np
import glob
import time
from tqdm import tqdm
import psutil

In [0]:
from pathlib import Path

import numpy as np

import glob

import time

from tqdm import tqdm

import psutil

# ----------------------------

# CONFIG (same as your tag logic)

# ----------------------------

cache_dir = Path("/dbfs/tmp/pftsleep_cache")

encoder_name = "PFTSleep"

num_files = 1229

frequency = 125

win_length = 750

hop_length = 750

max_seq_len_sec = 8 * 3600

def cache_tag(encoder_name, num_files, frequency, win_length, hop_length, max_seq_len_sec):

    return f"{encoder_name}__files{num_files}__freq{frequency}__win{win_length}__hop{hop_length}__max{max_seq_len_sec}"

tag = cache_tag(encoder_name, num_files, frequency, win_length, hop_length, max_seq_len_sec)

shard_dir = cache_dir / f"{tag}_shards"

shard_files = sorted(glob.glob(str(shard_dir / "Z_part_*.npy")))

assert len(shard_files) > 0, f"No shards found in {shard_dir}"

print(f"Found {len(shard_files)} shards")

print(f"RAM now: {psutil.virtual_memory().available / 1e9:.2f} GB free")

# ----------------------------

# 1) Compute total rows cheaply (mmap_mode avoids loading)

# ----------------------------

D = 512

total_rows = 0

first = np.load(shard_files[0], mmap_mode="r")

print("Example shard shape/dtype:", first.shape, first.dtype)

for f in shard_files:

    total_rows += np.load(f, mmap_mode="r").shape[0]

print(f"Total rows: {total_rows:,}  (expected ~5,899,200)")

print(f"Approx raw size float16: {total_rows * D * 2 / 1e9:.2f} GB")

print(f"Approx raw size float32: {total_rows * D * 4 / 1e9:.2f} GB")

# ----------------------------

# 2) Build a memmap on local NVMe (fast + avoids RAM ceilings)

# ----------------------------

local_dir = Path("/local_disk0/pftsleep_memmap")

local_dir.mkdir(parents=True, exist_ok=True)

mm_path = local_dir / f"X__{tag}__l2norm_f32.memmap"

shape_path = local_dir / f"X__{tag}__shape.txt"

# Create/overwrite memmap file

X_mm = np.memmap(mm_path, dtype=np.float32, mode="w+", shape=(total_rows, D))

# ----------------------------

# 3) Fill memmap sequentially (no vstack, no giant allocations)

# ----------------------------

t0 = time.time()

offset = 0

for f in tqdm(shard_files, desc="Writing memmap", unit="file"):

    shard = np.load(f)  # should be float16

    if shard.dtype != np.float16:

        # still fine; we cast below, but this warns you if storage isn't what you expect

        pass

    n = shard.shape[0]

    X_mm[offset:offset+n, :] = shard.astype(np.float32, copy=False)

    offset += n

X_mm.flush()

t1 = time.time()

with open(shape_path, "w") as s:

    s.write(f"{total_rows},{D}\n")

print(f"\nMemmap written: {mm_path}")

print(f"Write time: {t1 - t0:.2f} sec")

print(f"RAM now: {psutil.virtual_memory().available / 1e9:.2f} GB free")

# ----------------------------

# 4) In-place L2 normalize in chunks (still memmap-backed)

# ----------------------------

print("\nNormalizing memmap in chunks...")

chunk_rows = 250_000  # tune if you want (100k–500k is fine)

eps = 1e-8

t2 = time.time()

for start in tqdm(range(0, total_rows, chunk_rows), desc="L2 normalize", unit="chunk"):

    end = min(total_rows, start + chunk_rows)

    block = X_mm[start:end, :]  # view into memmap (does not load everything)

    norms = np.linalg.norm(block, axis=1, keepdims=True)

    block /= np.maximum(norms, eps)

X_mm.flush()

t3 = time.time()

print(f"Normalization time: {t3 - t2:.2f} sec")

print("✅ Memmap X is ready. Use X_mm like a normal array: X_mm[i:j]")

print(f"RAM now: {psutil.virtual_memory().available / 1e9:.2f} GB free")

# ----------------------------

# 5) Load your metadata normally (small)

# ----------------------------

night_id = np.load(cache_dir / f"night_id__{tag}.npy")

time_idx = np.load(cache_dir / f"time_idx__{tag}.npy")

zarr_file_idx = np.load(cache_dir / f"zarr_file_idx__{tag}.npy")

print("Metadata loaded:", night_id.shape, time_idx.shape, zarr_file_idx.shape)
 

In [0]:
# -------------------------
# Restructure demographics CSV to match zarr folder order
# -------------------------
import pandas as pd
from pathlib import Path

# Path to zarrs folder
zarrs_dir = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/zarrs")

# Get all zarr files in order
zarr_files = sorted(zarrs_dir.glob("*.zarr"))
print(f"📁 Found {len(zarr_files)} zarr files")

# Extract nsrrid from each zarr filename (e.g., "shhs1-200002.zarr" -> 200002)
zarr_nsrrids = []
for zarr_file in zarr_files:
    # Extract the number after "shhs1-"
    nsrrid = int(zarr_file.stem.split('-')[-1])
    zarr_nsrrids.append(nsrrid)
    print(f"  {zarr_file.name} -> nsrrid: {nsrrid}")

# Load the original demographics CSV
demographics_df = pd.read_csv('/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel.csv')
print(f"\n📊 Original demographics shape: {demographics_df.shape}")

# Create a mapping dataframe with the desired order
order_df = pd.DataFrame({'nsrrid': zarr_nsrrids, 'order': range(len(zarr_nsrrids))})

# Merge with demographics to get the order column
demographics_ordered = demographics_df.merge(order_df, on='nsrrid', how='inner')

# Sort by the order column
demographics_ordered = demographics_ordered.sort_values('order')

# Drop the order column
demographics_ordered = demographics_ordered.drop('order', axis=1)

print(f"\n✅ Restructured demographics shape: {demographics_ordered.shape}")
print(f"📋 Matched {len(demographics_ordered)} / {len(zarr_nsrrids)} zarr files")

# Show the new order
print(f"\n🔍 First 10 nsrrids in new order:")
print(demographics_ordered['nsrrid'].head(10).tolist())

# Save the restructured CSV
output_path = '/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel_ordered.csv'
demographics_ordered.to_csv(output_path, index=False)

print(f"\n💾 Saved restructured CSV to:")
print(f"   {output_path}")

# Verify the order matches
print(f"\n✅ Verification:")
print(f"   Zarr order: {zarr_nsrrids[:5]}")
print(f"   CSV order:  {demographics_ordered['nsrrid'].head(5).tolist()}")
print(f"   Match: {zarr_nsrrids[:5] == demographics_ordered['nsrrid'].head(5).tolist()}")

In [0]:
print("Total Shards:", len(list(shard_dir.glob("Z_part_*.npy"))))

In [0]:
# ------------------------------------------------------------

# 2) RANDOM HYPERPLANE LSH (TIGHTENED + SAMPLE-BASED)

# ------------------------------------------------------------
 
import numpy as np
 
# Use memmap-backed matrix

X = X_mm

N, D = X.shape
 
# -----------------------------------

# Use subsample for tuning

# -----------------------------------
 
rng = np.random.default_rng(42)

sample_size = min(5_000_000, N)

sample_idx = rng.choice(N, sample_size, replace=False)

X_sample = X[sample_idx]
 
print(f"Tuning LSH on sample of {sample_size:,} points")
 
# -----------------------------------

# LSH function

# -----------------------------------
 
def lsh_hash(X, n_bits, seed=42):

    rng = np.random.default_rng(seed)

    hyperplanes = rng.standard_normal((n_bits, D)).astype(np.float32)

    proj = X @ hyperplanes.T

    bits = (proj > 0)

    return np.packbits(bits, axis=1)
 
# -----------------------------------

# Bucket statistics

# -----------------------------------
 
def bucket_stats(hash_vals):

    # Convert bit arrays to compact representation

    unique, counts = np.unique(hash_vals, axis=0, return_counts=True)
 
    total = counts.sum()
 
    return {

        "nonempty": len(unique),

        "singleton_frac": float(np.sum(counts == 1)) / total,

        "median_size": float(np.median(counts)),

        "max_size": int(np.max(counts)),

    }
 
# -----------------------------------

# Bit depth search

# -----------------------------------
 
bits_range = range(6, 18)   # narrower, realistic range

target_buckets = 5000
 
best_bits = None

best_score = np.inf
 
for b in bits_range:
 
    h = lsh_hash(X_sample, n_bits=b)

    stats = bucket_stats(h)
 
    # Balanced scoring

    score = (

        abs(stats["nonempty"] - target_buckets)

        + 2000 * stats["singleton_frac"]

        + 0.001 * stats["max_size"]  # mild penalty for collapse

    )
 
    print(

        f"bits={b:2d} | "

        f"buckets={stats['nonempty']:4d} | "

        f"singleton_frac={stats['singleton_frac']:.3f} | "

        f"median={stats['median_size']:.1f} | "

        f"max={stats['max_size']}"

    )
 
    if score < best_score:

        best_score = score

        best_bits = b
 
print("\nChosen LSH bits:", best_bits)
 

In [0]:
# ------------------------------------------------------------
# 4) LSH-MEANS INITIALIZATION (TIGHTENED)
# ------------------------------------------------------------
 
from cuml.cluster import KMeans
from sklearn.preprocessing import normalize
import numpy as np
 
def lsh_means_init(X, n_clusters, n_bits, seed=42):
 
    N, D = X.shape
 
    print("Hashing full dataset...")
    h = lsh_hash(X, n_bits=n_bits, seed=seed)
 
    # Collapse packed bits into rows for grouping
    unique_hashes, inverse, counts = np.unique(
        h, axis=0, return_inverse=True, return_counts=True
    )
 
    print(f"Non-empty buckets: {len(unique_hashes)}")
 
    # Select largest buckets first
    order = np.argsort(counts)[::-1]
 
    # Take more than needed in case of tiny buckets
    candidate_idx = order[: max(n_clusters * 3, 100)]
 
    centroids = []
 
    for idx in candidate_idx:
        mask = (inverse == idx)
        if counts[idx] < 10:  # skip tiny buckets
            continue
        c = X[mask].mean(axis=0)
        centroids.append(c)
 
        if len(centroids) >= n_clusters:
            break
 
    if len(centroids) < n_clusters:
        raise RuntimeError("Not enough stable buckets to initialize centroids")
 
    C = np.vstack(centroids[:n_clusters])
 
    # L2 normalize centroids
    C = normalize(C, norm="l2")
 
    print(f"Initialized {C.shape[0]} centroids via LSH-means")
 
    return C.astype(np.float32)

In [0]:
# ------------------------------------------------------------
# FAST CORRECT K SWEEP
# ------------------------------------------------------------
 
from cuml.cluster import KMeans
from cuml.metrics.cluster import silhouette_score
import cupy as cp
import numpy as np
import time
 
X = X_mm
N, D = X.shape
 
# ----------------------------
# Sample once
# ----------------------------
 
rng = np.random.default_rng(42)
sample_size = min(400_000, N)
sample_idx = rng.choice(N, sample_size, replace=False)
 
X_sample = X[sample_idx]
X_sample_gpu = cp.asarray(X_sample, dtype=cp.float32)
 
print(f"Using {sample_size:,} points for K selection")
 
# ----------------------------
# Compute LSH ON SAMPLE ONLY
# ----------------------------
 
print("Computing LSH on sample only...")
h_sample = lsh_hash(X_sample, n_bits=best_bits)
 
unique_hashes, inverse, counts = np.unique(
    h_sample, axis=0, return_inverse=True, return_counts=True
)
 
bucket_order = np.argsort(counts)[::-1]
 
k_values = list(range(6, 15))
results = []
 
for k in k_values:
 
    start = time.time()
    print(f"\nTesting k={k}...")
 
    # Build centroids WITHOUT rehashing
    centroids = []
    for idx in bucket_order:
        if counts[idx] < 20:
            continue
        mask = (inverse == idx)
        centroids.append(X_sample[mask].mean(axis=0))
        if len(centroids) >= k:
            break
 
    C_gpu = cp.asarray(np.vstack(centroids[:k]), dtype=cp.float32)
 
    km = KMeans(
        n_clusters=k,
        init=C_gpu,
        max_iter=50,
        verbose=True,
        n_init=1,
        random_state=42,
    )
 
    labels = km.fit_predict(X_sample_gpu)
 
    # ---- GPU silhouette (NO .get()) ----
    sil = silhouette_score(X_sample_gpu, labels)
 
    elapsed = time.time() - start
    print(f"  silhouette={float(sil):.4f}  time={elapsed:.1f}s")
 
    results.append((k, float(sil)))
 
best_k, best_sil = max(results, key=lambda x: x[1])
 
print(f"\nChosen k={best_k} (silhouette={best_sil:.4f})")


In [0]:
best_k = 12

In [0]:
from cuml.cluster import KMeans
import cupy as cp
import numpy as np
import time
 
X = X_mm
N, D = X.shape
 
print("Moving full dataset to GPU...")
X_gpu = cp.asarray(X, dtype=cp.float32)
 
print("Computing LSH init on full dataset...")
 
# Hash full dataset once
h_full = lsh_hash(X, n_bits=best_bits)
 
unique_hashes, inverse, counts = np.unique(
    h_full, axis=0, return_inverse=True, return_counts=True
)
 
bucket_order = np.argsort(counts)[::-1]
 
centroids = []
for idx in bucket_order:
    if counts[idx] < 50:
        continue
    mask = (inverse == idx)
    centroids.append(X[mask].mean(axis=0))
    if len(centroids) >= best_k:
        break
 
C_gpu = cp.asarray(np.vstack(centroids[:best_k]), dtype=cp.float32)
 
print("Fitting final KMeans...")
t0 = time.time()
 
final_km = KMeans(
    n_clusters=best_k,
    init=C_gpu,
    max_iter=200,
    n_init=1,
    random_state=42,
)
 
cluster_id_gpu = final_km.fit_predict(X_gpu)
cluster_id = cp.asnumpy(cluster_id_gpu)
 
print(f"Final fit complete in {time.time() - t0:.1f}s")

In [0]:
import numpy as np
 
np.save(cache_dir / f"cluster_id__{tag}.npy", cluster_id)

print("Cluster IDs saved.")
 
unique, counts = np.unique(cluster_id, return_counts=True)
 
for u, c in zip(unique, counts):

    print(f"Cluster {u}: {c:,} windows")
 

In [0]:
# ============================================================

# UMAP Visualization of Clustered 6-Second Physiological States

# (2D scatter + density hexbin in one run)

# ============================================================
 
import numpy as np

import cupy as cp

import matplotlib.pyplot as plt

from cuml.manifold import UMAP
 
# ----------------------------

# 1) Subsample windows

# ----------------------------
 
rng = np.random.default_rng(42)

viz_size = min(200_000, len(cluster_id))  # safe range: 150k–250k
 
viz_idx = rng.choice(len(cluster_id), viz_size, replace=False)
 
X_viz = X_mm[viz_idx]

labels_viz = cluster_id[viz_idx]
 
print(f"Visualization sample size: {viz_size:,}")
 
# ----------------------------

# 2) Move to GPU

# ----------------------------
 
X_viz_gpu = cp.asarray(X_viz, dtype=cp.float32)
 
# ----------------------------

# 3) Run GPU UMAP

# ----------------------------
 
reducer = UMAP(

    n_neighbors=15,

    min_dist=0.1,

    n_components=2,

    metric="euclidean",

    random_state=42,

)
 
embedding_gpu = reducer.fit_transform(X_viz_gpu)

embedding = cp.asnumpy(embedding_gpu)
 
print("UMAP complete.")
 
# ----------------------------

# 4) 2D Scatter Plot

# ----------------------------
 
plt.figure(figsize=(10, 8))

scatter = plt.scatter(

    embedding[:, 0],

    embedding[:, 1],

    c=labels_viz,

    s=3,

    cmap="tab10",

    alpha=0.7,

)
 
plt.title(f"UMAP of 6-Second Physiological States (k={best_k})")

plt.xlabel("UMAP-1")

plt.ylabel("UMAP-2")

plt.colorbar(scatter, label="Cluster")

plt.tight_layout()

plt.show()
 
# ----------------------------

# 5) Density Hexbin Plot

# ----------------------------
 
plt.figure(figsize=(10, 8))

hb = plt.hexbin(

    embedding[:, 0],

    embedding[:, 1],

    gridsize=150,

    C=labels_viz,

    reduce_C_function=np.mean,

    cmap="viridis",

)
 
plt.colorbar(hb, label="Mean Cluster ID")

plt.title("UMAP Density View")

plt.xlabel("UMAP-1")

plt.ylabel("UMAP-2")

plt.tight_layout()

plt.show()
 

In [0]:
# ============================================================
# 3D UMAP Visualization (Interactive Plotly)
# ============================================================
 
import numpy as np
import cupy as cp
import plotly.graph_objects as go
from cuml.manifold import UMAP
 
# ----------------------------
# 1) Subsample windows
# ----------------------------
 
rng = np.random.default_rng(42)
viz_size = min(200_000, len(cluster_id))  # 150k–200k ideal
 
viz_idx = rng.choice(len(cluster_id), viz_size, replace=False)
 
X_viz = X_mm[viz_idx]
labels_viz = cluster_id[viz_idx]
 
print(f"Visualization sample size: {viz_size:,}")
 
# ----------------------------
# 2) Move to GPU
# ----------------------------
 
X_viz_gpu = cp.asarray(X_viz, dtype=cp.float32)
 
# ----------------------------
# 3) Run GPU UMAP (3D)
# ----------------------------
 
reducer = UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=3,   # 3D
    metric="euclidean",
    random_state=42,
)
 
embedding_gpu = reducer.fit_transform(X_viz_gpu)
embedding = cp.asnumpy(embedding_gpu)
 
print("3D UMAP complete.")
 
# ----------------------------
# 4) Interactive 3D Plot
# ----------------------------
 
fig = go.Figure(
    data=go.Scatter3d(
        x=embedding[:, 0],
        y=embedding[:, 1],
        z=embedding[:, 2],
        mode="markers",
        marker=dict(
            size=2,
            color=labels_viz,
            colorscale="Viridis",
            opacity=0.7,
            showscale=True,
            colorbar=dict(title="Cluster"),
        ),
    )
)
 
fig.update_layout(
    title=f"3D UMAP of 6-Second Physiological States (k={best_k})",
    height=800,
    scene=dict(
        xaxis_title="UMAP-1",
        yaxis_title="UMAP-2",
        zaxis_title="UMAP-3",
    ),
)
 
import plotly.io as piopio.show(fig, renderer='notebook')


In [0]:
import pandas as pd
 
df = pd.DataFrame({
    "cluster": cluster_id,
    "nsrrid": zarr_file_idx
})
 
cluster_fraction = (
    df.groupby("nsrrid")["cluster"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)

In [0]:
# -------------------------
# Restructure demographics CSV to match zarr folder order
# -------------------------
import pandas as pd
from pandas import Index

from pathlib import Path
 
# Path to zarrs folder
zarrs_dir = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/zarrs")
 
# Get all zarr files in order
zarr_files = sorted(zarrs_dir.glob("*.zarr"))
print(f"📁 Found {len(zarr_files)} zarr files")
 
# Extract nsrrid from each zarr filename (e.g., "shhs1-200002.zarr" -> 200002)
zarr_nsrrids = []
for zarr_file in zarr_files:
    # Extract the number after "shhs1-"
    nsrrid = int(zarr_file.stem.split('-')[-1])
    zarr_nsrrids.append(nsrrid)
    print(f"  {zarr_file.name} -> nsrrid: {nsrrid}")
 
# Load the original demographics CSV
demographics_df = pd.read_csv('/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel.csv')
print(f"\n📊 Original demographics shape: {demographics_df.shape}")
 
# Create a mapping dataframe with the desired order
order_df = pd.DataFrame({'nsrrid': zarr_nsrrids, 'order': range(len(zarr_nsrrids))})
 
# Merge with demographics to get the order column
demographics_ordered = demographics_df.merge(order_df, on='nsrrid', how='inner')
 
# Sort by the order column
demographics_ordered = demographics_ordered.sort_values('order')
 
# Drop the order column
demographics_ordered = demographics_ordered.drop('order', axis=1)
 
print(f"\n✅ Restructured demographics shape: {demographics_ordered.shape}")
print(f"📋 Matched {len(demographics_ordered)} / {len(zarr_nsrrids)} zarr files")
 
# Show the new order
print(f"\n🔍 First 10 nsrrids in new order:")
print(demographics_ordered['nsrrid'].head(10).tolist())
 
# Save the restructured CSV
output_path = '/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel_ordered.csv'
demographics_ordered.to_csv(output_path, index=False)
 
print(f"\n💾 Saved restructured CSV to:")
print(f"   {output_path}")
 
# Verify the order matches
print(f"\n✅ Verification:")
print(f"   Zarr order: {zarr_nsrrids[:5]}")
print(f"   CSV order:  {demographics_ordered['nsrrid'].head(5).tolist()}")
print(f"   Match: {zarr_nsrrids[:5] == demographics_ordered['nsrrid'].head(5).tolist()}")
 
Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype='int32', name='nsrrid')
 
# ------------------------------------------------------------

# FIX: Map zarr_file_idx -> real nsrrid

# ------------------------------------------------------------
 
from pathlib import Path
 
zarrs_dir = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/zarrs")

zarr_files = sorted(zarrs_dir.glob("*.zarr"))
 
zarr_nsrrids = [

    int(zarr_file.stem.split('-')[-1])

    for zarr_file in zarr_files

]
 
# Replace fake index (0..1228) with real nsrrid

cluster_fraction.index = zarr_nsrrids

cluster_fraction.index.name = "nsrrid"
 
print("Corrected index preview:")

print(cluster_fraction.index[:10])
 

In [0]:
# ------------------------------------------------------------

# Load demographics and merge

# ------------------------------------------------------------
 
import pandas as pd
 
demographics_df = pd.read_csv(

    "/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel_ordered.csv"

)
 
demographics_df["nsrrid"] = demographics_df["nsrrid"].astype(int)
 
cluster_demo = (

    cluster_fraction

    .reset_index()

    .merge(demographics_df, on="nsrrid", how="left")

)
 
print("Merged subjects:", len(cluster_demo))

print("Unique nsrrid:", cluster_demo["nsrrid"].nunique())
 

In [0]:
cluster_demo["dominant_cluster"] = (

    cluster_demo[cluster_fraction.columns]

    .idxmax(axis=1)

)
 

In [0]:
print(cluster_fraction.index[:10])

In [0]:
print(cluster_fraction.shape)
cluster_fraction.sum(axis=1).describe()
cluster_fraction.head()


output_path = Path("/Volumes/kumc_sleep/sleep_studies/shhs_data/clustering_outputs/")
cluster_fraction.to_csv(output_path / f"cluster_fraction__{tag}.csv")
print(f"Saved to: {output_path}")


In [0]:
import pandas as pd
 
demographics_path = "/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel_ordered.csv"
 
demographics_df = pd.read_csv(demographics_path)
 
print("Loaded demographics:", demographics_df.shape)
print(demographics_df.columns)

In [0]:
cluster_demo = cluster_fraction.reset_index().merge(

    demographics_df,

    on="nsrrid",

    how="left"

)
 
cluster_demo["dominant_cluster"] = cluster_fraction.idxmax(axis=1).values

dominant_map = cluster_demo[["nsrrid", "dominant_cluster"]]
 
output_path = Path("/Volumes/kumc_sleep/sleep_studies/shhs_data/clustering_outputs/") / f"dominant_cluster_map__{tag}.csv"
dominant_map.to_csv(output_path, index=False)
 
print(f"Saved dominant cluster map to: {output_path}")
 

In [0]:
cluster_demo.groupby("dominant_cluster").size()

In [0]:
import numpy as np
 
def summarize_cluster(df):
    return {
        "N": len(df),
        "Age_mean": df["age_s1"].mean(),
        "Age_min": df["age_s1"].min(),
        "Age_max": df["age_s1"].max(),
        "BMI_mean": df["bmi_s1"].mean(),
        "BMI_min": df["bmi_s1"].min(),
        "BMI_max": df["bmi_s1"].max(),
        "AHI_mean": df["ahi_a0h4_s1"].mean(),
        "AHI_min": df["ahi_a0h4_s1"].min(),
        "AHI_max": df["ahi_a0h4_s1"].max(),
        "ESS_mean": df["ess_s1"].mean(),
        "MinSat_mean": df["MinSat"].mean(),
        "PctLT90_mean": df["pctlt90"].mean(),
        "Diabetes_%": (df["Diabetes"] == "Yes").mean() * 100,
        "Prev_CVD_%": (df["prev_cvd_all_01"] == "Yes").mean() * 100,
        "Male_%": (df["gender"] == "Male").mean() * 100,
        "Smoker_%": (df["smokecat_s1"] == "Current").mean() * 100,
    }
 
summary_table = (
    cluster_demo
    .groupby("dominant_cluster")
    .apply(summarize_cluster)
    .apply(pd.Series)
)
 
summary_table
output_path = Path("/Volumes/kumc_sleep/sleep_studies/shhs_data/clustering_outputs/")
summary_table.to_csv(output_path / f"cluster_summary__{tag}.csv")
print(f"Saved to: {output_path}")
# ------------------------------------------------------------


In [0]:
# 8) UMAP ON RAW 512-D (NO PCA)
# ------------------------------------------------------------

reducer = UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=3,
    metric="euclidean",
    random_state=42,
    verbose=True
)
 
embedding = reducer.fit_transform(X)
 
fig = go.Figure(
    data=go.Scatter3d(
        x=embedding[:,0],
        y=embedding[:,1],
        z=embedding[:,2],
        mode="markers",
        marker=dict(
            size=3,
            color=cluster_id,
            showscale=True
        )
    )
)
 
fig.update_layout(
    title=f"RAW 512-D UMAP | k={best_k} | bits={best_bits}",
    height=800
)
 
display(fig)

In [0]:
# -------------------------
# Load demographics from sleep_excel and join with cluster assignments
# -------------------------
import pandas as pd

# Load the ordered demographics CSV
demographics_df = pd.read_csv('/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel_ordered.csv')

print(f"✅ Loaded demographics: {demographics_df.shape}")
print(f"\n📝 First few nsrrid values:")
print(demographics_df['nsrrid'].head(10))
print(f"\nnsrrid dtype: {demographics_df['nsrrid'].dtype}")

# Extract nsrrid from zarr_uid (e.g., "shhs1-200002" -> 200002)
# Use \d+$ to get all digits at the end of the string
cluster_mapping['nsrrid'] = cluster_mapping['zarr_uid'].str.extract(r'(\d+)$')[0].astype(int)

print(f"\n🔍 Extracted nsrrid from zarr_uid:")
print(cluster_mapping[['zarr_uid', 'nsrrid']].drop_duplicates())

# Ensure demographics nsrrid is int
demographics_df['nsrrid'] = demographics_df['nsrrid'].astype(int)

# Join cluster assignments with demographics
cluster_demo = cluster_mapping.merge(
    demographics_df[['nsrrid', 'age_s1', 'gender', 'race_s1', 'ethnicity_s1', 'smokecat_s1', 'Diabetes', 'COPD', 'bmi_s1']],
    on='nsrrid',
    how='left'
)

print(f"\n📊 Joined data shape: {cluster_demo.shape}")
print(f"✅ Successfully matched {cluster_demo['age_s1'].notna().sum()} / {len(cluster_demo)} records")

# Display sample
display(cluster_demo.head(20))

In [0]:
# -------------------------
# Analyze demographics by cluster
# -------------------------

# Define age ranges
def categorize_age(age):
    if pd.isna(age):
        return 'Unknown'
    elif age <= 18:
        return '0-18'
    elif age <= 38:
        return '19-38'
    elif age <= 57:
        return '39-57'
    elif age <= 76:
        return '58-76'
    elif age <= 95:
        return '77-95'
    else:
        return '96-114'

cluster_demo['age_range'] = cluster_demo['age_s1'].apply(categorize_age)

# Define BMI categories
def categorize_bmi(bmi):
    if pd.isna(bmi):
        return 'Unknown'
    elif bmi < 18.5:
        return 'Underweight (<18.5)'
    elif bmi < 25:
        return 'Healthy Weight (18.5-24.9)'
    elif bmi < 30:
        return 'Overweight (25-29.9)'
    elif bmi < 35:
        return 'Obesity Class I (30-34.9)'
    elif bmi < 40:
        return 'Obesity Class II (35-39.9)'
    else:
        return 'Obesity Class III (>=40)'

cluster_demo['bmi_category'] = cluster_demo['bmi_s1'].apply(categorize_bmi)

# Function to calculate percentages by cluster
def cluster_demographics(df, column, cluster_col='cluster_id'):
    """Calculate percentage distribution of a column within each cluster"""
    result = df.groupby([cluster_col, column]).size().unstack(fill_value=0)
    result_pct = result.div(result.sum(axis=1), axis=0) * 100
    return result_pct.round(2)

print("=" * 80)
print("DEMOGRAPHIC ANALYSIS BY CLUSTER")
print("=" * 80)

# Age Range Distribution
print("\n📊 AGE RANGE DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
age_dist = cluster_demographics(cluster_demo, 'age_range')
display(age_dist)

# Gender Distribution
print("\n👥 GENDER DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
gender_dist = cluster_demographics(cluster_demo, 'gender')
display(gender_dist)

# Race Distribution
print("\n🌍 RACE DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
race_dist = cluster_demographics(cluster_demo, 'race_s1')
display(race_dist)

# Ethnicity Distribution
print("\n🗺️ ETHNICITY DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
ethnicity_dist = cluster_demographics(cluster_demo, 'ethnicity_s1')
display(ethnicity_dist)

# Smoking Status Distribution
print("\n🚬 SMOKING STATUS DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
smoke_dist = cluster_demographics(cluster_demo, 'smokecat_s1')
display(smoke_dist)

# Diabetes Distribution
print("\n💉 DIABETES DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
diabetes_dist = cluster_demographics(cluster_demo, 'Diabetes')
display(diabetes_dist)

# COPD Distribution
print("\n🫁 COPD DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
copd_dist = cluster_demographics(cluster_demo, 'COPD')
display(copd_dist)

# BMI Category Distribution
print("\n⚖️ BMI CATEGORY DISTRIBUTION BY CLUSTER (%)")
print("-" * 80)
bmi_dist = cluster_demographics(cluster_demo, 'bmi_category')
display(bmi_dist)

# Summary statistics
print("\n📈 CLUSTER SUMMARY STATISTICS")
print("-" * 80)
summary = cluster_demo.groupby('cluster_id').agg({
    'nsrrid': 'count',
    'age_s1': ['mean', 'std', 'min', 'max'],
    'gender': [
        lambda x: (x == 'male').sum() / len(x) * 100,
        lambda x: (x == 'female').sum() / len(x) * 100
    ],
    'Diabetes': lambda x: (x == 'Yes').sum() / x.notna().sum() * 100 if x.notna().sum() > 0 else 0,
    'COPD': lambda x: (x == 'Yes').sum() / x.notna().sum() * 100 if x.notna().sum() > 0 else 0
}).round(2)
summary.columns = ['N', 'Age_Mean', 'Age_SD', 'Age_Min', 'Age_Max', 'Male_%', 'Female_%', 'Diabetes_%', 'COPD_%']
display(summary)

In [0]:
# -------------------------
# Enhanced 3D UMAP visualization with demographic hover info
# -------------------------
import plotly.graph_objects as go

# Aggregate cluster_demo to one row per window (take first occurrence per window_idx)
cluster_demo_unique = cluster_demo.drop_duplicates(subset='window_idx')

# Create hover text with all demographic info
hover_text = []
for idx, row in cluster_demo_unique.iterrows():
    text = (
        f"<b>nsrrid:</b> {row['nsrrid']}<br>"
        f"<b>Zarr File:</b> {row['zarr_uid']}<br>"
        f"<b>Cluster:</b> {row['cluster_id']}<br>"
        f"<b>Window:</b> {row['window_idx']}<br>"
        f"<b>Time:</b> {row['time_idx_sec']:.1f}s<br>"
        f"<br><b>Demographics:</b><br>"
        f"Age: {row['age_s1']} ({row['age_range']})<br>"
        f"Gender: {row['gender']}<br>"
        f"Race: {row['race_s1']}<br>"
        f"Ethnicity: {row['ethnicity_s1']}<br>"
        f"Smoking: {row['smokecat_s1']}<br>"
        f"Diabetes: {row['Diabetes']}<br>"
        f"COPD: {row['COPD']}"
    )
    hover_text.append(text)

# Create 3D scatter plot
fig = go.Figure(data=go.Scatter3d(
    x=embedding[:, 0],
    y=embedding[:, 1],
    z=embedding[:, 2],
    mode='markers',
    marker=dict(
        size=3,
        color=cluster_demo_unique['cluster_id'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Cluster ID"),
        line=dict(width=0)
    ),
    text=hover_text,
    hovertemplate='%{text}<extra></extra>',
    name='Data Points'
))

fig.update_layout(
    title=f"3D UMAP Visualization with Demographics | k={best_k} clusters",
    scene=dict(
        xaxis_title='UMAP 1',
        yaxis_title='UMAP 2',
        zaxis_title='UMAP 3',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.5)
        )
    ),
    height=800,
    width=1200,
    hovermode='closest'
)

display(fig)

print(f"\n✅ Interactive 3D UMAP created with {len(cluster_demo_unique)} points")
print(f"   Hover over points to see nsrrid and demographic information")

In [0]:
# -------------------------
# Save cluster assignments with demographics
# -------------------------

# Save the complete dataset
output_path_demo = cache_dir / f"cluster_assignments_with_demographics__{tag}.csv"
cluster_demo.to_csv(output_path_demo, index=False)

print(f"✅ Saved enhanced cluster assignments to:")
print(f"   {output_path_demo}")
print(f"\n📊 File contains {len(cluster_demo)} rows with:")
print(f"   - Cluster assignments")
print(f"   - Zarr file mapping")
print(f"   - Complete demographics (age, gender, race, ethnicity, smoking, diabetes, COPD)")

# GPU-Accelerated Clustering Complete ✅

## Workflow Summary

This notebook performs GPU-accelerated clustering on latent representations extracted by the **Representation_Extracting** notebook.

### Pipeline Steps

1. **Load Cached Latents** (Cell 5)
   - Loads `Z`, `night_id`, `time_idx`, `zarr_file_idx`, `windows_start_idx`, `zarr_files_list`
   - Loads `zarr_id_map.json` for file mapping

2. **Normalize** (Cell 7)
   - L2 normalization for angular geometry

3. **LSH Hashing** (Cell 8)
   - Random hyperplane LSH to find optimal bit depth
   - Targets ~2000 buckets for initialization

4. **LSH-Means Initialization** (Cell 9)
   - Uses LSH buckets to create smart initial centroids
   - **GPU-accelerated KMeans** for centroid refinement

5. **K-Value Sweep** (Cell 10)
   - Tests k values from 8-17
   - **GPU-accelerated KMeans** for each k
   - Evaluates using silhouette score

6. **Final Clustering** (Cell 11)
   - **GPU-accelerated KMeans** with best k
   - Assigns cluster IDs to all windows

7. **UMAP Visualization** (Cell 12)
   - 3D UMAP embedding colored by cluster

8. **Save Results** (Cell 13)
   - Creates comprehensive CSV with cluster assignments
   - Maps back to source zarr files
   - Includes all metadata for further analysis

## Output Files

**Saved to**: `cache_dir / cluster_assignments__{tag}.csv`

**Columns**:
- `window_idx`: Sequential window index
- `cluster_id`: Assigned cluster ID (from GPU KMeans)
- `zarr_file_idx`: Numeric zarr file ID
- `night_id`: Night/batch identifier
- `time_idx_sec`: Time index in seconds
- `windows_start_idx`: Start index in original zarr
- `zarr_uid`: Original zarr file UID
- `zarr_file_path`: Full path to source zarr file

## GPU Acceleration

✅ **All clustering operations use cuML GPU KMeans**
- Faster than CPU sklearn by 10-100x on large datasets
- Runs on T4 GPU (Standard_NC4as_T4_v3)
- Compatible with LSH-means initialization strategy

## Next Steps

Use the `cluster_assignments__{tag}.csv` file to:
- Analyze cluster characteristics
- Extract representative windows from each cluster
- Map clusters back to original zarr files for detailed inspection
- Perform downstream analysis on specific clusters